# 📖 Week 07 — Data Storytelling & Reporting (การเล่าเรื่องจากข้อมูล)

การวิเคราะห์ที่ดีไม่ได้จบแค่การรันตัวเลขหรือแปะรูปกราฟ แต่คือการสื่อสารข้อสรุปที่ **ตอบคำถาม**, **มีหลักฐานรองรับ**, และ **ระบุข้อจำกัดอย่างรับผิดชอบ**


## 📌 โครงสร้างการเล่าเรื่องแบบ 3 ส่วน (The 3-Sentence Rule)

1. **คำตอบตรงประเด็น (Observation / Key Answer):** บอกสิ่งที่ค้นพบหลักทันที ไม่เกริ่นโค้ดยาว
2. **หลักฐานเชิงประจักษ์ (Evidence):** อ้างอิงตัวเลขเฉลี่ย, สัดส่วน, หรือข้อมูลจากกราฟที่ตรวจสอบได้
3. **ข้อจำกัดของข้อมูล (Limitations):** ระบุขนาดตัวอย่าง (Sample Size), ค่าผิดปกติ หรือสิ่งที่ยังสรุปไม่ได้


In [1]:
import pandas as pd

# โหลดชุดข้อมูลนักเตะ
df = pd.read_csv("../data/footballers.csv")
display(df.head(3))

,name,club,league,nationality,position,age,height_cm,overall,pace,shooting,passing,dribbling,defending,physical,value_millions
0,Lionel Messi,Inter Miami,MLS,Argentina,RW,37.0,170.0,90,87,90,91,94,34,65,45
1,Cristiano Ronaldo,Al Nassr,Saudi Pro League,Portugal,ST,39.0,187.0,86,78,89,82,83,32,75,15
2,Kylian Mbappe,Real Madrid,La Liga,France,ST,26.0,178.0,91,97,88,80,92,36,78,180


---
## 1️⃣ ตัวอย่างการใช้งาน Methods สำหรับจัดเตรียมข้อมูลรายงาน
การจัดข้อมูลให้พร้อมสรุปผลด้วย `.groupby()`, `.agg()`, `.round()`, `.reset_index()`, และ `.iloc[]`


In [2]:
# 1.1 วิเคราะห์ค่า overall เฉลี่ยและจำนวนคนแยกตาม league
league_report = (
    df.groupby("league")["overall"]
    .agg(player_count="count", avg_overall="mean")
    .round(1)
    .sort_values(by="avg_overall", ascending=False)
    .reset_index()
)

print("=== ตารางสรุปสำหรับฝ่ายบริหาร (Executive Summary Table) ===")
display(league_report)

=== ตารางสรุปสำหรับฝ่ายบริหาร (Executive Summary Table) ===


,league,player_count,avg_overall
0,MLS,1,90.0
1,Bundesliga,4,87.2
2,Serie A,7,86.9
3,Ligue 1,3,86.0
4,Saudi Pro League,1,86.0
5,Premier League,19,80.4
6,La Liga,11,74.8


---
## 2️⃣ ตัวอย่างการใช้งาน Functions และ String Formatting สำหรับสร้างบทสรุปอัตโนมัติ
การใช้ Python `f-string`, `.iloc[]`, และตัวแปรสรุปผล เพื่อสร้างรายงานข้อความอัตโนมัติ


In [3]:
# 2.1 ดึงข้อมูลแถวอันดับ 1 และอันดับอื่นๆ มาสร้างประโยค
top_league = league_report.iloc[0]
sample_size = len(df)

# 2.2 สร้างข้อความสรุป 3 ส่วนอย่างรับผิดชอบ
sentence_1 = f"ในชุดข้อมูลนักเตะชั้นนำชุดนี้ ลีกที่มีคะแนนเฉลี่ยสูงสุดคือ {top_league['league']}"
sentence_2 = f"โดยมีคะแนน Overall เฉลี่ยอยู่ที่ {top_league['avg_overall']} คะแนน (จากนักเตะในลีกนี้ {int(top_league['player_count'])} คน)"
sentence_3 = f"อย่างไรก็ตาม ข้อมูลนี้มีนักเตะเพียง {sample_size} คนใน 5 ลีกหลัก จึงสะท้อนเฉพาะกลุ่มนักเตะระดับท็อป ไม่สามารถเหมารวมผู้เล่นทุกคนในลีกได้"

full_story = f"""
📢 บทสรุปผลการวิเคราะห์ (Data Storytelling Summary):
1. {sentence_1}
2. {sentence_2}
3. {sentence_3}
"""
print(full_story)


📢 บทสรุปผลการวิเคราะห์ (Data Storytelling Summary):
1. ในชุดข้อมูลนักเตะชั้นนำชุดนี้ ลีกที่มีคะแนนเฉลี่ยสูงสุดคือ MLS
2. โดยมีคะแนน Overall เฉลี่ยอยู่ที่ 90.0 คะแนน (จากนักเตะในลีกนี้ 1 คน)
3. อย่างไรก็ตาม ข้อมูลนี้มีนักเตะเพียง 46 คนใน 5 ลีกหลัก จึงสะท้อนเฉพาะกลุ่มนักเตะระดับท็อป ไม่สามารถเหมารวมผู้เล่นทุกคนในลีกได้



---
## 3️⃣ Custom Function สำหรับสร้างสรุปรายงานอัตโนมัติ
ฟังก์ชัน `generate_story_report()` สำหรับนำไปใช้ซ้ำกับคอลัมน์อื่นๆ ได้อย่างรวดเร็ว


In [4]:
def generate_story_report(data, group_col, metric_col):
    """
    ฟังก์ชันวิเคราะห์กลุ่มข้อมูลและพิมพ์สรุปผล 3 มิติอัตโนมัติ
    """
    summary = (
        data.groupby(group_col)[metric_col]
        .agg(count="count", mean="mean")
        .sort_values("mean", ascending=False)
        .round(1)
        .reset_index()
    )
    best = summary.iloc[0]
    total = len(data)
    
    print(f"=== รายงานวิเคราะห์ {metric_col} แยกตาม {group_col} ===")
    display(summary)
    print(f"👉 สรุป: กลุ่ม '{best[group_col]}' มีค่าเฉลี่ย {metric_col} สูงสุดที่ {best['mean']} (กลุ่มตัวอย่าง {int(best['count'])} จากทั้งหมด {total} คน)")
    return summary

# ทดลองใช้งาน Custom Function
pos_story = generate_story_report(df, group_col="position", metric_col="pace")

=== รายงานวิเคราะห์ pace แยกตาม position ===


,position,count,mean
0,RW,4,82.2
1,ST,8,81.1
2,LW,6,80.0
3,CAM,6,79.3
4,RB,2,73.0
5,CM,6,70.8
6,LB,2,68.0
7,CB,6,64.5
8,CDM,3,63.7
9,GK,3,20.0


👉 สรุป: กลุ่ม 'RW' มีค่าเฉลี่ย pace สูงสุดที่ 82.2 (กลุ่มตัวอย่าง 4 จากทั้งหมด 46 คน)
